# Debt Regime Analysis Robustness Checks 

In [1]:
import datetime

# ===== GLOBAL VARIABLES =====
RELOAD_DATA = False 
TODAY = str(datetime.datetime.now().date())
DEBT_TO_GDP_PERCENTILE = 67 
# ============================

In [2]:
# ===== IMPORTS =============
from ResearchFramework.ResearchHandler import ResearchHandler
from ResearchFramework.simulation import *
import ResearchFramework.transforms as tf
from loader import load_fred_master
from macro_scores import score
import statsmodels.api as sm
# ============================

# ====== INIT ================
if RELOAD_DATA:
    _ = load_fred_master()
fred = ResearchHandler("data/fred_master.csv", handler=score)
print(f"Today is {TODAY}.")
fred.data = fred.data[fred.data["date"]<=TODAY]
fred.data.tail()
thresh = fred.data["Debt_to_GDP"].quantile(DEBT_TO_GDP_PERCENTILE / 100)
# ============================

Today is 2026-03-28.


In [3]:
# ===== ROBUSTNESS CHECKS =====
# Framework: each check returns a dict with standardized keys
# so we can collect them into a summary table at the end.

import numpy as np
import warnings
warnings.filterwarnings("ignore")

def run_inflation_model(data, regime_col="Regime", dv="CPI_YoY",
                        ivs=["M2_YoY_Lag_78w", "CPI_Energy_YoY", 
                             "Inflation_Momentum", "Expect_Anchor_Signed"],
                        threshold=None, threshold_col="Debt_to_GDP"):
    """
    Estimate regime-conditional inflation models and return
    standardized results dict.
    
    If threshold is provided, re-splits the data at that value.
    Otherwise uses the existing Regime column.
    """
    df = data.copy()
    
    if threshold is not None:
        df["Regime"] = np.where(df[threshold_col] > threshold, "High", "Low")
        regime_col = "Regime"
    
    out = {}
    for label in ["Low", "High"]:
        sub = df[df[regime_col] == label].dropna(subset=ivs + [dv])
        X = sm.add_constant(sub[ivs])
        y = sub[dv]
        model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 26})
        
        out[label] = {
            "n": len(y),
            "r2": model.rsquared,
            "coefficients": dict(model.params),
            "pvalues": dict(model.pvalues),
            "resid_std": model.resid.std(),
        }
    
    # Summary metrics
    m2_low = out["Low"]["coefficients"].get("M2_YoY_Lag_78w", np.nan)
    m2_high = out["High"]["coefficients"].get("M2_YoY_Lag_78w", np.nan)
    exp_low = out["Low"]["coefficients"].get("Expect_Anchor_Signed", np.nan)
    exp_high = out["High"]["coefficients"].get("Expect_Anchor_Signed", np.nan)
    
    out["summary"] = {
        "M2_Low": m2_low,
        "M2_High": m2_high,
        "Sign_Flip": (m2_low < 0) and (m2_high > 0),
        "M2_Swing": m2_high - m2_low,
        "Expect_Low": exp_low,
        "Expect_High": exp_high,
        "Expect_Ratio": exp_high / exp_low if abs(exp_low) > 1e-6 else np.nan,
        "n_Low": out["Low"]["n"],
        "n_High": out["High"]["n"],
    }
    return out

In [4]:
# ── ROBUSTNESS 1: Threshold Sensitivity ──────────────────────────
print("=" * 60)
print("ROBUSTNESS 1: Threshold Sensitivity")
print("=" * 60)

threshold_results = {}
for pct in [60, 65, 70, 75, 80, 85, 90]:
    thresh_val = fred.data["Debt_to_GDP"].quantile(pct / 100)
    result = run_inflation_model(fred.data, threshold=thresh_val)
    threshold_results[pct] = result["summary"]
    s = result["summary"]
    flip = "✓" if s["Sign_Flip"] else "✗"
    print(f"  P{pct} (thresh={thresh_val:.1f}%): "
          f"M2 Low={s['M2_Low']:+.3f}, M2 High={s['M2_High']:+.3f}, "
          f"Flip={flip}, n_High={s['n_High']}")

threshold_df = pd.DataFrame(threshold_results).T
threshold_df.index.name = "Percentile"

ROBUSTNESS 1: Threshold Sensitivity
  P60 (thresh=94.5%): M2 Low=-0.130, M2 High=+0.083, Flip=✓, n_High=756
  P65 (thresh=99.8%): M2 Low=-0.128, M2 High=+0.099, Flip=✓, n_High=652
  P70 (thresh=101.3%): M2 Low=-0.127, M2 High=+0.118, Flip=✓, n_High=561
  P75 (thresh=102.9%): M2 Low=-0.125, M2 High=+0.119, Flip=✓, n_High=470
  P80 (thresh=104.3%): M2 Low=-0.126, M2 High=+0.122, Flip=✓, n_High=378
  P85 (thresh=117.4%): M2 Low=+0.073, M2 High=+0.093, Flip=✗, n_High=273
  P90 (thresh=119.5%): M2 Low=+0.095, M2 High=+0.066, Flip=✗, n_High=181


In [5]:
# ── ROBUSTNESS 2: Alternative Dependent Variables ────────────────
print("=" * 60)
print("ROBUSTNESS 2: Alternative DVs")
print("=" * 60)

alt_dv_results = {}
for dv in ["CPI_YoY", "Core_PCE_YoY", "Core_CPI_YoY"]:
    result = run_inflation_model(fred.data, dv=dv, threshold=thresh)
    alt_dv_results[dv] = result["summary"]
    s = result["summary"]
    flip = "✓" if s["Sign_Flip"] else "✗"
    print(f"  DV={dv:<15s}: M2 Low={s['M2_Low']:+.3f}, M2 High={s['M2_High']:+.3f}, "
          f"Flip={flip}, Expect ratio={s['Expect_Ratio']:.1f}×")

ROBUSTNESS 2: Alternative DVs
  DV=CPI_YoY        : M2 Low=-0.129, M2 High=+0.104, Flip=✓, Expect ratio=4.6×
  DV=Core_PCE_YoY   : M2 Low=-0.073, M2 High=+0.068, Flip=✓, Expect ratio=6.7×
  DV=Core_CPI_YoY   : M2 Low=-0.105, M2 High=+0.096, Flip=✓, Expect ratio=3.0×


In [6]:
# ── ROBUSTNESS 3: Additional Controls ────────────────────────────
print("=" * 60)
print("ROBUSTNESS 3: Additional Controls")
print("=" * 60)

base_ivs = ["M2_YoY_Lag_78w", "CPI_Energy_YoY", "Inflation_Momentum", "Expect_Anchor_Signed"]
control_specs = {
    "Base (no controls)":       base_ivs,
    "+ Unemployment":           base_ivs + ["Unemployment_Rate"],
    "+ Home Prices":            base_ivs + ["Home_Price_YoY_Lag_78w"],
    "+ UE + Home Prices":       base_ivs + ["Unemployment_Rate", "Home_Price_YoY_Lag_78w"],
}

control_results = {}
for spec_name, ivs in control_specs.items():
    result = run_inflation_model(fred.data, ivs=ivs, threshold=thresh)
    control_results[spec_name] = result["summary"]
    s = result["summary"]
    flip = "✓" if s["Sign_Flip"] else "✗"
    print(f"  {spec_name:<25s}: M2 Low={s['M2_Low']:+.3f}, M2 High={s['M2_High']:+.3f}, "
          f"Flip={flip}, M2 swing={s['M2_Swing']:+.3f}")

ROBUSTNESS 3: Additional Controls
  Base (no controls)       : M2 Low=-0.129, M2 High=+0.104, Flip=✓, M2 swing=+0.233
  + Unemployment           : M2 Low=-0.137, M2 High=+0.107, Flip=✓, M2 swing=+0.245
  + Home Prices            : M2 Low=-0.132, M2 High=+0.063, Flip=✓, M2 swing=+0.195
  + UE + Home Prices       : M2 Low=-0.137, M2 High=+0.067, Flip=✓, M2 swing=+0.204


In [7]:
# ── ROBUSTNESS 4: Quarterly Frequency ────────────────────────────
print("=" * 60)
print("ROBUSTNESS 4: Quarterly Frequency (every 13th obs)")
print("=" * 60)

quarterly = fred.data.iloc[::13].copy()
quarterly["Regime"] = np.where(quarterly["Debt_to_GDP"] > thresh, "High", "Low")
result = run_inflation_model(quarterly, threshold=thresh)
s = result["summary"]
flip = "✓" if s["Sign_Flip"] else "✗"
print(f"  M2 Low={s['M2_Low']:+.3f}, M2 High={s['M2_High']:+.3f}, Flip={flip}")
print(f"  n_Low={s['n_Low']}, n_High={s['n_High']}")
print(f"  Expect Low={s['Expect_Low']:+.3f}, Expect High={s['Expect_High']:+.3f}, Ratio={s['Expect_Ratio']:.1f}×")
quarterly_result = s

ROBUSTNESS 4: Quarterly Frequency (every 13th obs)
  M2 Low=-0.119, M2 High=+0.105, Flip=✓
  n_Low=89, n_High=47
  Expect Low=+0.427, Expect High=+1.605, Ratio=3.8×


In [8]:
# ── ROBUSTNESS 5: Exclude 2020-2021 ─────────────────────────────
print("=" * 60)
print("ROBUSTNESS 5: Exclude COVID Spike (2020-2021)")
print("=" * 60)

no_covid = fred.data[
    ~((fred.data["date"].dt.year >= 2020) & (fred.data["date"].dt.year <= 2021))
].copy()
no_covid["Regime"] = np.where(no_covid["Debt_to_GDP"] > thresh, "High", "Low")
result = run_inflation_model(no_covid, threshold=thresh)
s = result["summary"]
flip = "✓" if s["Sign_Flip"] else "✗"
print(f"  M2 Low={s['M2_Low']:+.3f}, M2 High={s['M2_High']:+.3f}, Flip={flip}")
print(f"  n_Low={s['n_Low']}, n_High={s['n_High']}")
print(f"  Expect Low={s['Expect_Low']:+.3f}, Expect High={s['Expect_High']:+.3f}, Ratio={s['Expect_Ratio']:.1f}×")
no_covid_result = s

ROBUSTNESS 5: Exclude COVID Spike (2020-2021)
  M2 Low=-0.129, M2 High=+0.107, Flip=✓
  n_Low=1148, n_High=508
  Expect Low=+0.354, Expect High=+1.738, Ratio=4.9×


In [9]:
# ── ROBUSTNESS 6: Bootstrap Coefficient Confidence ───────────────
print("=" * 60)
print("ROBUSTNESS 6: Bootstrap M2 Coefficient (1000 iterations)")
print("=" * 60)

n_boot = 1000
boot_m2_low = []
boot_m2_high = []
boot_exp_low = []
boot_exp_high = []

ivs = ["M2_YoY_Lag_78w", "CPI_Energy_YoY", "Inflation_Momentum", "Expect_Anchor_Signed"]
dv = "CPI_YoY"

low_data = fred.data[fred.data["Debt_to_GDP"] <= thresh]
high_data = fred.data[fred.data["Debt_to_GDP"] > thresh]

for regime_label, regime_data in [("Low", low_data), ("High", high_data)]:
    clean = regime_data.dropna(subset=ivs + [dv])
    
    for i in range(n_boot):
        sample = clean.sample(n=len(clean), replace=True, random_state=i)
        X = sm.add_constant(sample[ivs])
        y = sample[dv]
        try:
            model = sm.OLS(y, X).fit(disp=0)
            m2_coef = model.params["M2_YoY_Lag_78w"]
            exp_coef = model.params["Expect_Anchor_Signed"]
            if regime_label == "Low":
                boot_m2_low.append(m2_coef)
                boot_exp_low.append(exp_coef)
            else:
                boot_m2_high.append(m2_coef)
                boot_exp_high.append(exp_coef)
        except:
            pass

boot_m2_low = np.array(boot_m2_low)
boot_m2_high = np.array(boot_m2_high)
boot_m2_diff = boot_m2_high[:len(boot_m2_low)] - boot_m2_low[:len(boot_m2_high)]

print(f"\n  M2 coefficient — Low regime:")
print(f"    Mean: {boot_m2_low.mean():+.4f}, 95% CI: [{np.percentile(boot_m2_low, 2.5):+.4f}, {np.percentile(boot_m2_low, 97.5):+.4f}]")
print(f"  M2 coefficient — High regime:")
print(f"    Mean: {boot_m2_high.mean():+.4f}, 95% CI: [{np.percentile(boot_m2_high, 2.5):+.4f}, {np.percentile(boot_m2_high, 97.5):+.4f}]")
print(f"  M2 coefficient DIFFERENCE (High - Low):")
print(f"    Mean: {boot_m2_diff.mean():+.4f}, 95% CI: [{np.percentile(boot_m2_diff, 2.5):+.4f}, {np.percentile(boot_m2_diff, 97.5):+.4f}]")
print(f"    CI excludes zero: {'✓' if np.percentile(boot_m2_diff, 2.5) > 0 else '✗'}")

boot_exp_diff = np.array(boot_exp_high[:len(boot_exp_low)]) - np.array(boot_exp_low[:len(boot_exp_high)])
print(f"\n  Expect coefficient DIFFERENCE (High - Low):")
print(f"    Mean: {boot_exp_diff.mean():+.4f}, 95% CI: [{np.percentile(boot_exp_diff, 2.5):+.4f}, {np.percentile(boot_exp_diff, 97.5):+.4f}]")
print(f"    CI excludes zero: {'✓' if np.percentile(boot_exp_diff, 2.5) > 0 else '✗'}")

ROBUSTNESS 6: Bootstrap M2 Coefficient (1000 iterations)

  M2 coefficient — Low regime:
    Mean: -0.1284, 95% CI: [-0.1393, -0.1177]
  M2 coefficient — High regime:
    Mean: +0.1036, 95% CI: [+0.0915, +0.1171]
  M2 coefficient DIFFERENCE (High - Low):
    Mean: +0.2320, 95% CI: [+0.2166, +0.2477]
    CI excludes zero: ✓

  Expect coefficient DIFFERENCE (High - Low):
    Mean: +1.2680, 95% CI: [+1.0859, +1.4479]
    CI excludes zero: ✓


In [10]:
# ── SUMMARY TABLE ────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ROBUSTNESS SUMMARY")
print("=" * 70)

summary_rows = []

# R1: Threshold sensitivity
for pct, s in threshold_results.items():
    summary_rows.append({
        "Test": f"Threshold P{pct}",
        "M2_Low": s["M2_Low"], "M2_High": s["M2_High"],
        "Sign_Flip": "✓" if s["Sign_Flip"] else "✗",
        "Expect_Ratio": f"{s['Expect_Ratio']:.1f}×",
        "n_High": s["n_High"],
    })

# R2: Alt DVs
for dv_name, s in alt_dv_results.items():
    summary_rows.append({
        "Test": f"DV: {dv_name}",
        "M2_Low": s["M2_Low"], "M2_High": s["M2_High"],
        "Sign_Flip": "✓" if s["Sign_Flip"] else "✗",
        "Expect_Ratio": f"{s['Expect_Ratio']:.1f}×",
        "n_High": s["n_High"],
    })

# R3: Controls
for spec_name, s in control_results.items():
    summary_rows.append({
        "Test": f"Controls: {spec_name}",
        "M2_Low": s["M2_Low"], "M2_High": s["M2_High"],
        "Sign_Flip": "✓" if s["Sign_Flip"] else "✗",
        "Expect_Ratio": f"{s['Expect_Ratio']:.1f}×",
        "n_High": s["n_High"],
    })

# R4: Quarterly
summary_rows.append({
    "Test": "Quarterly frequency",
    "M2_Low": quarterly_result["M2_Low"], "M2_High": quarterly_result["M2_High"],
    "Sign_Flip": "✓" if quarterly_result["Sign_Flip"] else "✗",
    "Expect_Ratio": f"{quarterly_result['Expect_Ratio']:.1f}×",
    "n_High": quarterly_result["n_High"],
})

# R5: No COVID
summary_rows.append({
    "Test": "Exclude 2020-2021",
    "M2_Low": no_covid_result["M2_Low"], "M2_High": no_covid_result["M2_High"],
    "Sign_Flip": "✓" if no_covid_result["Sign_Flip"] else "✗",
    "Expect_Ratio": f"{no_covid_result['Expect_Ratio']:.1f}×",
    "n_High": no_covid_result["n_High"],
})

# R6: Bootstrap
summary_rows.append({
    "Test": "Bootstrap CI excludes 0",
    "M2_Low": boot_m2_low.mean(), "M2_High": boot_m2_high.mean(),
    "Sign_Flip": "✓" if np.percentile(boot_m2_diff, 2.5) > 0 else "✗",
    "Expect_Ratio": f"—",
    "n_High": "—",
})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format="%.3f"))

# Overall pass rate
total = len([r for r in summary_rows if r["Sign_Flip"] in ["✓", "✗"]])
passed = len([r for r in summary_rows if r["Sign_Flip"] == "✓"])
print(f"\nSign flip survives: {passed}/{total} tests")


ROBUSTNESS SUMMARY
                        Test  M2_Low  M2_High Sign_Flip Expect_Ratio n_High
               Threshold P60  -0.130    0.083         ✓         4.4×    756
               Threshold P65  -0.128    0.099         ✓         4.5×    652
               Threshold P70  -0.127    0.118         ✓         4.6×    561
               Threshold P75  -0.125    0.119         ✓         3.7×    470
               Threshold P80  -0.126    0.122         ✓         3.4×    378
               Threshold P85   0.073    0.093         ✗         0.9×    273
               Threshold P90   0.095    0.066         ✗         0.8×    181
                 DV: CPI_YoY  -0.129    0.104         ✓         4.6×    613
            DV: Core_PCE_YoY  -0.073    0.068         ✓         6.7×    613
            DV: Core_CPI_YoY  -0.105    0.096         ✓         3.0×    613
Controls: Base (no controls)  -0.129    0.104         ✓         4.6×    613
    Controls: + Unemployment  -0.137    0.107         ✓         8.8×

In [11]:
# ── Threshold Search: Find the regime switch point ───────────────
print("=" * 60)
print("THRESHOLD SEARCH: M2 Sign Flip Boundary")
print("=" * 60)

search_results = []
for pct in range(50, 91):
    thresh_val = fred.data["Debt_to_GDP"].quantile(pct / 100)
    result = run_inflation_model(fred.data, threshold=thresh_val)
    s = result["summary"]
    search_results.append({
        "percentile": pct,
        "threshold": thresh_val,
        "M2_Low": s["M2_Low"],
        "M2_High": s["M2_High"],
        "Sign_Flip": s["Sign_Flip"],
        "M2_High_p": result["High"]["pvalues"].get("M2_YoY_Lag_78w", np.nan),
        "n_High": s["n_High"],
    })

search_df = pd.DataFrame(search_results)

# Find the boundary
first_flip = search_df[search_df["Sign_Flip"] == True].iloc[0]
last_no_flip = search_df[search_df["Sign_Flip"] == False].iloc[-1] if not search_df["Sign_Flip"].all() else None

print(f"\n  First percentile with sign flip: P{int(first_flip['percentile'])} "
      f"(Debt/GDP = {first_flip['threshold']:.1f}%)")
if last_no_flip is not None:
    print(f"  Last percentile WITHOUT flip:    P{int(last_no_flip['percentile'])} "
          f"(Debt/GDP = {last_no_flip['threshold']:.1f}%)")
print(f"\n  Boundary region: ~{first_flip['threshold']:.0f}% Debt/GDP")

# Print the transition zone
print(f"\n  {'Pctl':>4s}  {'Debt/GDP':>8s}  {'M2_Low':>7s}  {'M2_High':>8s}  {'Flip':>4s}  {'p_High':>7s}  {'n_High':>6s}")
print("  " + "-" * 55)
for _, row in search_df.iterrows():
    marker = "  ←" if row["percentile"] == first_flip["percentile"] else ""
    print(f"  P{int(row['percentile']):>2d}  {row['threshold']:>8.1f}  {row['M2_Low']:>+7.3f}  {row['M2_High']:>+8.3f}  "
          f"{'✓' if row['Sign_Flip'] else '✗':>4s}  {row['M2_High_p']:>7.4f}  {int(row['n_High']):>6d}{marker}")

THRESHOLD SEARCH: M2 Sign Flip Boundary

  First percentile with sign flip: P50 (Debt/GDP = 65.0%)
  Last percentile WITHOUT flip:    P90 (Debt/GDP = 119.5%)

  Boundary region: ~65% Debt/GDP

  Pctl  Debt/GDP   M2_Low   M2_High  Flip   p_High  n_High
  -------------------------------------------------------
  P50      65.0   -0.100    +0.080     ✓   0.0028     939  ←
  P51      65.3   -0.102    +0.080     ✓   0.0030     926
  P52      73.2   -0.109    +0.080     ✓   0.0034     900
  P53      77.1   -0.109    +0.077     ✓   0.0047     887
  P54      82.4   -0.123    +0.084     ✓   0.0029     861
  P55      84.0   -0.120    +0.085     ✓   0.0028     848
  P56      88.1   -0.120    +0.081     ✓   0.0033     822
  P57      89.6   -0.127    +0.082     ✓   0.0024     809
  P58      92.2   -0.132    +0.082     ✓   0.0021     782
  P59      93.0   -0.135    +0.082     ✓   0.0018     770
  P60      94.5   -0.130    +0.083     ✓   0.0026     756
  P61      97.1   -0.130    +0.087     ✓   0.0028

In [13]:
# ── Threshold Search Visualization ────────────────────────────
# Run this after the threshold search cell in the robustness notebook.
# Requires search_df to be in scope.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.7, 0.3],
    subplot_titles=["M2 Coefficient by Debt/GDP Threshold", "p-value (High Debt Regime)"],
)

# ── Top panel: M2 coefficients for both regimes ──────────────

# Low-debt M2 coefficient
fig.add_trace(go.Scatter(
    x=search_df["threshold"],
    y=search_df["M2_Low"],
    mode="lines+markers",
    name="M2 coef (Low Debt)",
    line=dict(color="#636EFA", width=2),
    marker=dict(size=4),
    hovertemplate="Debt/GDP: %{x:.1f}%<br>M2 Low: %{y:.3f}<extra></extra>",
), row=1, col=1)

# High-debt M2 coefficient
fig.add_trace(go.Scatter(
    x=search_df["threshold"],
    y=search_df["M2_High"],
    mode="lines+markers",
    name="M2 coef (High Debt)",
    line=dict(color="#EF553B", width=2),
    marker=dict(size=4),
    hovertemplate="Debt/GDP: %{x:.1f}%<br>M2 High: %{y:.3f}<extra></extra>",
), row=1, col=1)

# Zero reference line
fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1, row=1, col=1)

# 100% Debt/GDP vertical marker
fig.add_vline(
    x=100, line_dash="dash", line_color="#00CC96", line_width=2,
    row=1, col=1,
)
fig.add_annotation(
    x=100, y=search_df["M2_High"].max() + 0.01,
    text="100% Debt/GDP",
    showarrow=False,
    font=dict(size=11, color="#00CC96"),
    xref="x", yref="y",
)

# Shade the sign-flip failure zone
fig.add_vrect(
    x0=117, x1=120, fillcolor="red", opacity=0.06, line_width=0,
    row=1, col=1,
)
fig.add_annotation(
    x=118.5, y=-0.08,
    text="Sign flip<br>fails here",
    showarrow=False,
    font=dict(size=9, color="#EF553B"),
    xref="x", yref="y",
)

# ── Bottom panel: p-value ─────────────────────────────────────

fig.add_trace(go.Scatter(
    x=search_df["threshold"],
    y=search_df["M2_High_p"],
    mode="lines+markers",
    name="p-value (M2 High)",
    line=dict(color="#AB63FA", width=2),
    marker=dict(size=4),
    showlegend=True,
    hovertemplate="Debt/GDP: %{x:.1f}%<br>p-value: %{y:.4f}<extra></extra>",
), row=2, col=1)

# Significance threshold
fig.add_hline(y=0.05, line_dash="dot", line_color="gray", line_width=1, row=2, col=1)
fig.add_annotation(
    x=search_df["threshold"].iloc[-1], y=0.05,
    text="p = 0.05",
    showarrow=False,
    font=dict(size=9, color="gray"),
    xanchor="right",
    xref="x2", yref="y2",
)

# 100% marker on bottom panel too
fig.add_vline(
    x=100, line_dash="dash", line_color="#00CC96", line_width=2,
    row=2, col=1,
)

fig.update_layout(
    template="plotly_white",
    font=dict(family="Inter, system-ui, sans-serif", size=13),
    margin=dict(l=60, r=30, t=60, b=50),
    height=700,
    width=1000,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center",
    ),
)

fig.update_xaxes(title_text="Debt/GDP Threshold (%)", row=2, col=1)
fig.update_yaxes(title_text="M2 Coefficient (β)", row=1, col=1)
fig.update_yaxes(title_text="p-value", row=2, col=1, type="log")

fig.show()
fig.write_image("charts/threshold_search.png", scale=2)